In [1]:
import pandas as pd
import plotly.express as px

In [2]:
df = pd.read_json("../../opendata-basketball/data/matches/191313/191313_tracking_data.jsonl.gz", lines=True)
last = df[(df["period"] == df["period"].max()) & (df["gameClock"] <= 120)]
last = last[last["homePlayers"].str.len() > 0].iloc[::5]  # tracked frames only, 5 per second

In [3]:
rows = []
for r in last.itertuples():
    for team, players in [("home", r.homePlayers), ("away", r.awayPlayers)]:
        for p in players:
            rows.append((r.frameIdx, team, p["jersey"], p["xyz"][0], p["xyz"][1]))
    rows.append((r.frameIdx, "ball", "", r.ball["xyz"][0], r.ball["xyz"][1]))
points = pd.DataFrame(rows, columns=["frameIdx", "team", "jersey", "x", "y"])

In [4]:
fig = px.scatter(points, x="x", y="y", color="team", text="jersey", animation_frame="frameIdx",
                 range_x=[-55, 55], range_y=[-32, 32], width=900, height=600)
fig.update_traces(marker_size=18, textfont_color="white")
fig.add_shape(type="rect", x0=-45.93, x1=45.93, y0=-24.61, y1=24.61)  # 28m x 15m court, in feet
fig.update_yaxes(scaleanchor="x")
play = fig.layout.updatemenus[0].buttons[0].args[1]
play["frame"]["duration"] = 200  # real-time speed
play["transition"] = {"duration": 200, "easing": "linear"}
fig